# Semana 3 – Mejora de Modelos y Selección

Esta sección implementa:
- Cross-Validation (validación cruzada estratificada)
- Hyperparameter Tuning con GridSearchCV y RandomizedSearchCV
- Comparación formal de modelos optimizados
- Análisis de sesgo/varianza (overfitting vs underfitting)

## 1. Importaciones y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

## 2. Carga de datos y separación

Se divide el dataset en tres conjuntos estratificados:
- **Entrenamiento** (60 %): ajuste de parámetros del modelo.
- **Validación** (20 %): verificación del rendimiento durante la optimización de hiperparámetros.
- **Prueba** (20 %): evaluación final, nunca vista durante entrenamiento ni optimización.


In [ ]:
df = pd.read_csv('data/processed/Datos_clean.csv')

X = df.drop(columns=['Class'])
y = df['Class']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print(f'Entrenamiento : {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Validación    : {X_val.shape[0]} muestras ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Prueba        : {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.0f}%)')


## 3. Funciones de utilidad reutilizables

In [ ]:
def evaluar_modelo(nombre, y_true, y_pred, y_prob):
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall    = recall_score(y_true, y_pred)
    f1        = f1_score(y_true, y_pred)
    auc_roc   = roc_auc_score(y_true, y_prob)

    print(f'  MÉTRICAS – {nombre.upper()}')
    print('='*42)
    print(f'  Accuracy  : {accuracy:.4f}')
    print(f'  Precision : {precision:.4f}')
    print(f'  Recall    : {recall:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'  AUC-ROC   : {auc_roc:.4f}')
    print('\nReporte completo:')
    print(classification_report(y_true, y_pred, target_names=['Benigno (0)', 'Maligno (1)']))

    return {
        'Modelo': nombre, 'Accuracy': accuracy, 'Precision': precision,
        'Recall': recall, 'F1-Score': f1, 'AUC-ROC': auc_roc
    }


def plot_confusion_matrix(nombre, y_true, y_pred, cmap='Blues'):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=['Benigno (0)', 'Maligno (1)'],
                yticklabels=['Benigno (0)', 'Maligno (1)'])
    plt.title(f'Matriz de Confusión – {nombre}')
    plt.ylabel('Clase Real')
    plt.xlabel('Clase Predicha')
    plt.tight_layout()
    plt.show()
    tn, fp, fn, tp = cm.ravel()
    print(f'Verdaderos Negativos (benigno correcto): {tn}')
    print(f'Falsos Positivos (benigno como maligno): {fp}')
    print(f'Falsos Negativos (maligno como benigno): {fn}  ← crítico en contexto médico')
    print(f'Verdaderos Positivos (maligno correcto): {tp}')


def plot_roc_curve(nombre, y_true, y_prob, color='steelblue'):
    auc = roc_auc_score(y_true, y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, color=color, lw=2, label=f'AUC = {auc:.4f}')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Clasificador aleatorio')
    plt.xlabel('Tasa de Falsos Positivos')
    plt.ylabel('Tasa de Verdaderos Positivos (Recall)')
    plt.title(f'Curva ROC – {nombre}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()


## 5. Hyperparameter Tuning

Se aplica **GridSearchCV** para Regresión Logística y SVM, y **RandomizedSearchCV** para Random Forest y XGBoost. En todos los casos se optimiza AUC-ROC con StratifiedKFold k=5 sobre el conjunto de entrenamiento. Al finalizar cada búsqueda se reporta además el AUC-ROC sobre el **conjunto de validación** como confirmación independiente.


In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

modelos_base = {
    'Regresión Logística': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'XGBoost':             xgb.XGBClassifier(
                               scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
                               n_estimators=100, max_depth=5, learning_rate=0.1,
                               eval_metric='logloss', random_state=42),
    'SVM (RBF)':           SVC(kernel='rbf', C=1.0, gamma='scale',
                               class_weight='balanced', probability=True, random_state=42)
}

print('='*60)
print('CROSS-VALIDATION (StratifiedKFold, k=10) – MODELOS BASE')
print('='*60)

resultados_cv_base = {}
for nombre, modelo in modelos_base.items():
    X_cv = X_train_scaled if 'SVM' in nombre else X_train

    scores_auc = cross_val_score(modelo, X_cv, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    scores_f1  = cross_val_score(modelo, X_cv, y_train, cv=cv, scoring='f1', n_jobs=-1)
    scores_rec = cross_val_score(modelo, X_cv, y_train, cv=cv, scoring='recall', n_jobs=-1)

    resultados_cv_base[nombre] = {
        'AUC-ROC': scores_auc,
        'F1':      scores_f1,
        'Recall':  scores_rec
    }

    print(f'\n{nombre}:')
    print(f'  AUC-ROC : {scores_auc.mean():.4f} ± {scores_auc.std():.4f}')
    print(f'  F1-Score: {scores_f1.mean():.4f}  ± {scores_f1.std():.4f}')
    print(f'  Recall  : {scores_rec.mean():.4f}  ± {scores_rec.std():.4f}')


### Visualización de distribución de AUC-ROC por fold

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
data_plot = [resultados_cv_base[m]['AUC-ROC'] for m in modelos_base]
labels    = list(modelos_base.keys())

bp = ax.boxplot(data_plot, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
colores = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('AUC-ROC')
ax.set_title('Distribución de AUC-ROC – Cross-Validation (k=10)')
ax.set_ylim(0.95, 1.01)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning

Se aplica **GridSearchCV** (búsqueda exhaustiva) para Regresión Logística y SVM,
y **RandomizedSearchCV** (búsqueda aleatoria eficiente) para Random Forest y XGBoost.
En todos los casos se optimiza sobre AUC-ROC con StratifiedKFold k=5.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('\n[1/4] Optimizando Regresión Logística...')

param_grid_lr = {
    'C':       [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver':  ['liblinear', 'saga']
}

grid_lr = GridSearchCV(
    LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
    param_grid_lr, cv=cv5, scoring='roc_auc', n_jobs=-1, verbose=0
)
grid_lr.fit(X_train, y_train)
lr_opt = grid_lr.best_estimator_

val_auc_lr = roc_auc_score(y_val, lr_opt.predict_proba(X_val)[:, 1])
print(f'  Mejores parámetros: {grid_lr.best_params_}')
print(f'  Mejor AUC-ROC (CV train): {grid_lr.best_score_:.4f}')
print(f'  AUC-ROC (val set):        {val_auc_lr:.4f}')


print('\n[2/4] Optimizando Random Forest...')

param_dist_rf = {
    'n_estimators':      [50, 100, 200, 300, 500],
    'max_depth':         [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', None]
}

rand_rf = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_dist_rf, n_iter=50, cv=cv5, scoring='roc_auc',
    n_jobs=-1, random_state=42, verbose=0
)
rand_rf.fit(X_train, y_train)
rf_opt = rand_rf.best_estimator_

val_auc_rf = roc_auc_score(y_val, rf_opt.predict_proba(X_val)[:, 1])
print(f'  Mejores parámetros: {rand_rf.best_params_}')
print(f'  Mejor AUC-ROC (CV train): {rand_rf.best_score_:.4f}')
print(f'  AUC-ROC (val set):        {val_auc_rf:.4f}')


print('\n[3/4] Optimizando XGBoost...')

scale_w = (y_train==0).sum() / (y_train==1).sum()

param_dist_xgb = {
    'n_estimators':     [50, 100, 200, 300],
    'max_depth':        [3, 4, 5, 6, 8],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha':        [0, 0.1, 0.5, 1.0],
    'reg_lambda':       [1, 1.5, 2.0]
}

rand_xgb = RandomizedSearchCV(
    xgb.XGBClassifier(scale_pos_weight=scale_w, eval_metric='logloss', random_state=42),
    param_dist_xgb, n_iter=50, cv=cv5, scoring='roc_auc',
    n_jobs=-1, random_state=42, verbose=0
)
rand_xgb.fit(X_train, y_train)
xgb_opt = rand_xgb.best_estimator_

val_auc_xgb = roc_auc_score(y_val, xgb_opt.predict_proba(X_val)[:, 1])
print(f'  Mejores parámetros: {rand_xgb.best_params_}')
print(f'  Mejor AUC-ROC (CV train): {rand_xgb.best_score_:.4f}')
print(f'  AUC-ROC (val set):        {val_auc_xgb:.4f}')


print('\n[4/4] Optimizando SVM...')

param_grid_svm = {
    'C':     [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]
}

grid_svm = GridSearchCV(
    SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42),
    param_grid_svm, cv=cv5, scoring='roc_auc', n_jobs=-1, verbose=0
)
grid_svm.fit(X_train_scaled, y_train)
svm_opt = grid_svm.best_estimator_

val_auc_svm = roc_auc_score(y_val, svm_opt.predict_proba(X_val_scaled)[:, 1])
print(f'  Mejores parámetros: {grid_svm.best_params_}')
print(f'  Mejor AUC-ROC (CV train): {grid_svm.best_score_:.4f}')
print(f'  AUC-ROC (val set):        {val_auc_svm:.4f}')


## 6. Evaluación de modelos optimizados en test set

Se evalúan los cuatro modelos con sus mejores hiperparámetros sobre el conjunto
de prueba (20% del dataset, nunca visto durante entrenamiento u optimización).

In [ ]:
y_pred_lr_opt  = lr_opt.predict(X_test)
y_prob_lr_opt  = lr_opt.predict_proba(X_test)[:, 1]

y_pred_rf_opt  = rf_opt.predict(X_test)
y_prob_rf_opt  = rf_opt.predict_proba(X_test)[:, 1]

y_pred_xgb_opt = xgb_opt.predict(X_test)
y_prob_xgb_opt = xgb_opt.predict_proba(X_test)[:, 1]

y_pred_svm_opt = svm_opt.predict(X_test_scaled)
y_prob_svm_opt = svm_opt.predict_proba(X_test_scaled)[:, 1]

print('\n' + '='*60)
print('RESULTADOS EN CONJUNTO DE PRUEBA – MODELOS OPTIMIZADOS')
print('='*60)

res_lr  = evaluar_modelo('Regresión Logística (opt)', y_test, y_pred_lr_opt, y_prob_lr_opt)
res_rf  = evaluar_modelo('Random Forest (opt)',       y_test, y_pred_rf_opt, y_prob_rf_opt)
res_xgb = evaluar_modelo('XGBoost (opt)',             y_test, y_pred_xgb_opt, y_prob_xgb_opt)
res_svm = evaluar_modelo('SVM (opt)',                 y_test, y_pred_svm_opt, y_prob_svm_opt)


### Matrices de confusión – modelos optimizados

In [ ]:
modelos_opt_pred = [
    ('Regresión Logística (opt)', y_pred_lr_opt,  'Blues'),
    ('Random Forest (opt)',       y_pred_rf_opt,  'Greens'),
    ('XGBoost (opt)',             y_pred_xgb_opt, 'Oranges'),
    ('SVM (opt)',                 y_pred_svm_opt, 'Purples'),
]
for nombre, y_pred, cmap in modelos_opt_pred:
    plot_confusion_matrix(nombre, y_test, y_pred, cmap=cmap)

### Curvas ROC comparativas

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
configs = [
    ('Regresión Logística (opt)', y_prob_lr_opt,  'steelblue'),
    ('Random Forest (opt)',       y_prob_rf_opt,  'green'),
    ('XGBoost (opt)',             y_prob_xgb_opt, 'orange'),
    ('SVM (opt)',                 y_prob_svm_opt, 'purple'),
]
for nombre, y_prob, color in configs:
    auc = roc_auc_score(y_test, y_prob)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{nombre} (AUC={auc:.4f})')

ax.plot([0,1], [0,1], 'k--', label='Clasificador aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title('Curvas ROC – Comparación de Modelos Optimizados')
ax.legend(loc='lower right', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Cross-Validation sobre modelos optimizados

In [ ]:
modelos_opt = {
    'Reg. Logística (opt)': (lr_opt,  X_train),
    'Random Forest (opt)':  (rf_opt,  X_train),
    'XGBoost (opt)':        (xgb_opt, X_train),
    'SVM (opt)':            (svm_opt, X_train_scaled),
}

print('='*60)
print('CROSS-VALIDATION (k=10) – MODELOS OPTIMIZADOS')
print('='*60)

resultados_cv_opt = {}
for nombre, (modelo, X_cv) in modelos_opt.items():
    scores_auc = cross_val_score(modelo, X_cv, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    scores_f1  = cross_val_score(modelo, X_cv, y_train, cv=cv, scoring='f1',      n_jobs=-1)
    scores_rec = cross_val_score(modelo, X_cv, y_train, cv=cv, scoring='recall',  n_jobs=-1)

    resultados_cv_opt[nombre] = {
        'AUC-ROC': scores_auc, 'F1': scores_f1, 'Recall': scores_rec
    }
    print(f'\n{nombre}:')
    print(f'  AUC-ROC : {scores_auc.mean():.4f} ± {scores_auc.std():.4f}')
    print(f'  F1-Score: {scores_f1.mean():.4f}  ± {scores_f1.std():.4f}')
    print(f'  Recall  : {scores_rec.mean():.4f}  ± {scores_rec.std():.4f}')


## 8. Análisis de Overfitting vs Underfitting (Curvas de Aprendizaje)

Las curvas de aprendizaje muestran cómo evoluciona el rendimiento en entrenamiento
y validación a medida que aumenta el tamaño del conjunto de entrenamiento. Una
brecha grande entre ambas curvas indica **overfitting**; si ambas son bajas, indica
**underfitting**.

In [ ]:
def plot_learning_curve(estimator, X, y, nombre, color='steelblue', cv=5):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv, scoring='roc_auc',
        train_sizes=np.linspace(0.1, 1.0, 10),
        n_jobs=-1, random_state=42
    )
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_mean, 'o-', color=color, label='Entrenamiento')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                     alpha=0.1, color=color)
    plt.plot(train_sizes, val_mean, 's--', color='tomato', label='Validación (CV)')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                     alpha=0.1, color='tomato')
    plt.xlabel('Tamaño del conjunto de entrenamiento')
    plt.ylabel('AUC-ROC')
    plt.title(f'Curva de Aprendizaje – {nombre}')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.ylim(0.88, 1.02)
    plt.tight_layout()
    plt.show()

    gap_final = train_mean[-1] - val_mean[-1]
    print(f'{nombre}:')
    print(f'  AUC entrenamiento (final): {train_mean[-1]:.4f}')
    print(f'  AUC validación   (final): {val_mean[-1]:.4f}')
    print(f'  Brecha train-val:          {gap_final:.4f}', end=' ')
    if gap_final > 0.05:
        print('← posible overfitting')
    elif val_mean[-1] < 0.95:
        print('← posible underfitting')
    else:
        print('← buen balance sesgo/varianza')
    print()


print('Generando curvas de aprendizaje...\n')
plot_learning_curve(lr_opt,  X_train,        y_train, 'Regresión Logística (opt)', 'steelblue')
plot_learning_curve(rf_opt,  X_train,        y_train, 'Random Forest (opt)',       'green')
plot_learning_curve(xgb_opt, X_train,        y_train, 'XGBoost (opt)',             'orange')
plot_learning_curve(svm_opt, X_train_scaled, y_train, 'SVM (opt)',                 'purple')


## 9. Tabla comparativa final

In [ ]:
resultados = [res_lr, res_rf, res_xgb, res_svm]
df_comparacion = pd.DataFrame(resultados).round(4).set_index('Modelo')

print('\n' + '='*70)
print('TABLA COMPARATIVA FINAL – MODELOS OPTIMIZADOS (Conjunto de Prueba)')
print('='*70)
print(df_comparacion.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metricas = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']

x     = np.arange(len(metricas))
width = 0.2
colores_mod = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
nombres_mod = [r.replace(' (opt)', '') for r in df_comparacion.index]

for i, (nombre_largo, color) in enumerate(zip(df_comparacion.index, colores_mod)):
    valores = [df_comparacion.loc[nombre_largo, m] for m in metricas]
    axes[0].bar(x + i * width, valores, width, label=nombres_mod[i],
                color=color, alpha=0.85)

axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(metricas, rotation=15, ha='right')
axes[0].set_ylim(0.88, 1.02)
axes[0].set_title('Comparación de Métricas – Modelos Optimizados')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

sns.heatmap(df_comparacion[metricas].astype(float), annot=True, fmt='.4f',
            cmap='YlGnBu', ax=axes[1], vmin=0.88, vmax=1.0,
            cbar_kws={'label': 'Valor'})
axes[1].set_title('Heatmap de Métricas – Modelos Optimizados')
axes[1].set_xticklabels(metricas, rotation=15, ha='right')

plt.tight_layout()
plt.show()


## 10. Modelo final seleccionado

Con base en la comparación rigurosa, se selecciona **Random Forest optimizado**
como el modelo final, dado que:
- Alcanza Recall = 1.00 (cero falsos negativos), crítico en diagnóstico médico
- Mantiene AUC-ROC ≥ 0.99 tanto en CV como en test set
- Exhibe bajo overfitting según curvas de aprendizaje
- Ofrece importancia de características nativa para interpretabilidad clínica

In [ ]:
print('\n' + '='*60)
print('MODELO FINAL SELECCIONADO: RANDOM FOREST (OPTIMIZADO)')
print('='*60)
print(f'Mejores hiperparámetros: {rand_rf.best_params_}')
print()

importances = pd.DataFrame({
    'Variable':    X.columns,
    'Importancia': rf_opt.feature_importances_
}).sort_values('Importancia', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importances, x='Importancia', y='Variable',
            hue='Variable', palette='Greens_r', legend=False)
plt.title('Importancia de Características – Random Forest Final')
plt.xlabel('Importancia (Gini)')
plt.tight_layout()
plt.show()

print(importances.to_string(index=False))
